In [ ]:
import mlflow

EXPERIMENT_NAME = "risco_credito_comparacao_modelos"

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

if mlflow.active_run() is not None:
   mlflow.end_run()

import mlflow.sklearn
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score    
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def gerar_base_credito(n_amostras: int = 5000, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    n = n_amostras

    idade = rng.integers(22, 66, size=n)
    renda_mensal = rng.lognormal(mean=np.log(3500), sigma=0.45, size=n).round(2)
    tempo_emprego_anos = rng.exponential(scale=4, size=n).clip(0, 35).round(1)
    score_interno = rng.normal(650, 90, size=n).clip(300, 900).astype(int)
    valor_solicitado = rng.lognormal(mean=np.log(8000), sigma=0.6, size=n).round(2)
    num_parcelas = rng.choice([6, 12, 18, 24, 36, 48], size=n, p=[0.1, 0.25, 0.2, 0.2, 0.15, 0.1])
    tipo_renda = rng.choice(["CLT", "autonomo", "aposentado"], size=n, p=[0.65, 0.22, 0.13])
    possui_restricao = rng.choice(["sim", "nao"], size=n, p=[0.12, 0.88])

    # Probabilidade base de default (logit simplificado)
    z = (
        -3.2
        - 0.025 * (score_interno - 650)
        - 0.00008 * renda_mensal
        + 0.00002 * valor_solicitado
        + 0.015 * (num_parcelas / 12)
        - 0.04 * tempo_emprego_anos
    )
    z = np.asarray(z, dtype=float)
    z += np.where(possui_restricao == "sim", 0.85, 0)
    z += np.where(tipo_renda == "autonomo", 0.35, 0)
    z += rng.normal(0, 0.35, size=n)
    p_default = 1 / (1 + np.exp(-z))
    inadimplente = rng.binomial(1, p_default)

    return pd.DataFrame(
        {
            "idade": idade,
            "renda_mensal": renda_mensal,
            "tempo_emprego_anos": tempo_emprego_anos,
            "score_interno": score_interno,
            "valor_solicitado": valor_solicitado,
            "num_parcelas": num_parcelas,
            "tipo_renda": tipo_renda,
            "possui_restricao": possui_restricao,
            "inadimplente": inadimplente,
        }
    )


df = gerar_base_credito(6000)
df.head()

In [ ]:
df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.boxplot(column="score_interno", by="inadimplente", ax=axes[0])
axes[0].set_title("Score interno vs inadimplência")
axes[0].set_xlabel("inadimplente")
df.boxplot(column="renda_mensal", by="inadimplente", ax=axes[1])
axes[1].set_title("Renda mensal vs inadimplência")
axes[1].set_xlabel("inadimplente")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
TARGET = "inadimplente"
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

num_cols = ["idade", "renda_mensal", "tempo_emprego_anos", "score_interno", "valor_solicitado", "num_parcelas"]
cat_cols = ["tipo_renda", "possui_restricao"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ]
)

X_train.shape, X_test.shape

In [ ]:
clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)

model_pipe = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", clf),
    ]
)

model_pipe.fit(X_train, y_train)
y_proba = model_pipe.predict_proba(X_test)[:, 1]
y_pred = model_pipe.predict(X_test)

print("ROC-AUC (teste):", round(roc_auc_score(y_test, y_proba), 4))


 4.x

In [ ]:
def make_preprocess():
   numeric_pipe = Pipeline(
       steps=[
           ("imputer", SimpleImputer(strategy="median")),
           ("scaler", StandardScaler()),
       ]
   )

   categorical_pipe = Pipeline(
       steps=[
           ("imputer", SimpleImputer(strategy="most_frequent")),
           ("onehot", OneHotEncoder(handle_unknown="ignore")),
       ]
   )

   return ColumnTransformer(
       transformers=[
           ("num", numeric_pipe, num_cols),
           ("cat", categorical_pipe, cat_cols),
       ]
   )


In [ ]:
model_candidates = {
   "logistic_regression": LogisticRegression(
       max_iter=2000,
       class_weight="balanced",
       random_state=RANDOM_STATE,
   ),
   "decision_tree": DecisionTreeClassifier(
       max_depth=5,
       min_samples_leaf=50,
       class_weight="balanced",
       random_state=RANDOM_STATE,
   ),
   "random_forest": RandomForestClassifier(
       n_estimators=100,
       max_depth=8,
       min_samples_leaf=20,
       class_weight="balanced",
       n_jobs=-1,
       random_state=RANDOM_STATE,
   ),
}


In [ ]:
results = []
CV_FOLDS = 3

for model_name, estimator in model_candidates.items():
   print(f"Treinando: {model_name}")

   # Mantemos a validação cruzada fora do autolog para evitar runs ruidosas.
   mlflow.sklearn.autolog(disable=True)

   cv_pipe = Pipeline(
       steps=[
           ("prep", make_preprocess()),
           ("model", clone(estimator)),
       ]
   )

   cv = StratifiedKFold(
       n_splits=CV_FOLDS,
       shuffle=True,
       random_state=RANDOM_STATE,
   )

   cv_scores = cross_val_score(
       cv_pipe,
       X_train,
       y_train,
       cv=cv,
       scoring="roc_auc",
       n_jobs=1,
   )

   # Ativamos autolog apenas para o treino final desta run.
   mlflow.sklearn.autolog(
       log_input_examples=False,
       log_model_signatures=True,
       log_models=True,
       silent=True,
   )

   run_name = f"aula04_autolog_{model_name}"

   with mlflow.start_run(run_name=run_name) as run:
       pipe = Pipeline(
           steps=[
               ("prep", make_preprocess()),
               ("model", clone(estimator)),
           ]
       )

       pipe.fit(X_train, y_train)

       y_proba = pipe.predict_proba(X_test)[:, 1]
       y_pred = pipe.predict(X_test)

       metrics = {
           "roc_auc_teste": float(roc_auc_score(y_test, y_proba)),
           "roc_auc_cv_media": float(cv_scores.mean()),
           "roc_auc_cv_desvio": float(cv_scores.std()),
           "accuracy_teste": float(accuracy_score(y_test, y_pred)),
           "precision_inadimplente": float(
               precision_score(y_test, y_pred, zero_division=0)
           ),
           "recall_inadimplente": float(
               recall_score(y_test, y_pred, zero_division=0)
           ),
           "f1_inadimplente": float(
               f1_score(y_test, y_pred, zero_division=0)
           ),
       }

       mlflow.log_param("model_name", model_name)
       mlflow.log_param("cv_folds", CV_FOLDS)
       mlflow.log_param("dataset_version", "synthetic_credit_v1")
       mlflow.log_param("target", TARGET)

       mlflow.log_metrics(metrics)

       mlflow.set_tags(
           {
               "project": "risco_credito",
               "course_lesson": "aula_04",
               "tracking_type": "autolog_plus_manual_metrics",
               "task": "binary_classification",
               "model_name": model_name,
               "run_purpose": "compare_model_candidates",
           }
       )

       mlflow.log_dict(
           {
               "model_name": model_name,
               "run_id": run.info.run_id,
               "metrics": metrics,
               "notes": (
                   "Run gerada com mlflow.sklearn.autolog e métricas "
                   "de teste registradas manualmente para comparação."
               ),
           },
           "reports/evaluation_summary.json",
       )

       results.append(
           {
               "run_id": run.info.run_id,
               "run_name": run_name,
               "model_name": model_name,
               **metrics,
           }
       )

   # Desativamos depois da run para evitar logging automático acidental.
   mlflow.sklearn.autolog(disable=True)

comparison_df = (
   pd.DataFrame(results)
   .sort_values("roc_auc_teste", ascending=False)
   .reset_index(drop=True)
)

comparison_df


Continuação  4

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
   experiment_ids=[experiment.experiment_id],
   filter_string="tags.course_lesson = 'aula_04'",
   order_by=["metrics.roc_auc_teste DESC"],
)

runs.shape


In [ ]:
comparison_cols = [
   "run_id",
   "tags.mlflow.runName",
   "params.model_name",
   "metrics.roc_auc_teste",
   "metrics.roc_auc_cv_media",
   "metrics.roc_auc_cv_desvio",
   "metrics.accuracy_teste",
   "metrics.precision_inadimplente",
   "metrics.recall_inadimplente",
   "metrics.f1_inadimplente",
]

available_cols = [col for col in comparison_cols if col in runs.columns]

ranking = (
   runs[available_cols]
   .sort_values("metrics.roc_auc_teste", ascending=False)
   .reset_index(drop=True)
)

ranking


In [ ]:
shortlist = mlflow.search_runs(
   experiment_ids=[experiment.experiment_id],
   filter_string=(
       "tags.course_lesson = 'aula_04' "
       "and metrics.roc_auc_teste >= 0.85 "
       "and metrics.recall_inadimplente >= 0.85"
   ),
   order_by=["metrics.roc_auc_teste DESC"],
)

shortlist_cols = [col for col in comparison_cols if col in shortlist.columns]

shortlist[shortlist_cols]


In [ ]:
ranking_with_roles = ranking.copy()

ranking_with_roles["candidate_role"] = [
   "champion_candidate" if i == 0 else "challenger"
   for i in range(len(ranking_with_roles))
]

ranking_with_roles


In [ ]:
ranking.to_csv("comparacao_runs_aula04.csv", index=False)
ranking


5.x

In [ ]:
import subprocess


def get_git_commit(default="not_available"):
   try:
       commit = subprocess.check_output(
           ["git", "rev-parse", "--short", "HEAD"],
           text=True,
       ).strip()
       return commit
   except Exception:
       return default


CODE_VERSION = get_git_commit()
CODE_VERSION


In [ ]:
COMMON_TAGS = {
   "project": "risco_credito",
   "task": "binary_classification",
   "target": "inadimplente",
   "dataset_version": "synthetic_credit_v1",
   "feature_set": "core_features_v1",
   "split_strategy": "train_test_stratified_75_25",
   "code_version": CODE_VERSION,
}


In [ ]:
EXPERIMENT_NAME = "risco_credito_comparacao_modelos"

mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs = mlflow.search_runs(
   experiment_ids=[experiment.experiment_id],
   filter_string="tags.course_lesson = 'aula_04'",
   order_by=["metrics.roc_auc_teste DESC"],
)

if runs.empty:
   raise ValueError("Nenhuma run da Aula 4 foi encontrada para seleção.")

runs.shape


In [ ]:
runs

In [ ]:
model_col = (
   "params.model_name"
   if "params.model_name" in runs.columns
   else "tags.model_name"
)

decision_cols = [
   "run_id",
   "tags.mlflow.runName",
   model_col,
   "metrics.roc_auc_teste",
   "metrics.roc_auc_cv_media",
   "metrics.roc_auc_cv_desvio",
   "metrics.accuracy_teste",
   "metrics.precision_inadimplente",
   "metrics.recall_inadimplente",
   "metrics.f1_inadimplente",
]

available_cols = [col for col in decision_cols if col in runs.columns]

decision_df = runs[available_cols].copy()

decision_df = decision_df.rename(
   columns={
       "tags.mlflow.runName": "run_name",
       model_col: "model_name",
       "metrics.roc_auc_teste": "roc_auc_teste",
       "metrics.roc_auc_cv_media": "roc_auc_cv_media",
       "metrics.roc_auc_cv_desvio": "roc_auc_cv_desvio",
       "metrics.accuracy_teste": "accuracy_teste",
       "metrics.precision_inadimplente": "precision_inadimplente",
       "metrics.recall_inadimplente": "recall_inadimplente",
       "metrics.f1_inadimplente": "f1_inadimplente",
   }
)

metric_cols = [
   "roc_auc_teste",
   "roc_auc_cv_media",
   "roc_auc_cv_desvio",
   "accuracy_teste",
   "precision_inadimplente",
   "recall_inadimplente",
   "f1_inadimplente",
]

decision_df[metric_cols] = decision_df[metric_cols].astype(float)

decision_df.sort_values("roc_auc_teste", ascending=False).shape


In [ ]:
selection_policy = {
   "technical_score": {
       "roc_auc_teste": 0.35,
       "recall_inadimplente": 0.25,
       "precision_inadimplente": 0.15,
       "f1_inadimplente": 0.15,
       "stability_score": 0.10,
   },
   "final_score": {
       "technical_score": 0.75,
       "interpretability_score": 0.15,
       "operational_simplicity_score": 0.10,
   },
   "notes": (
       "Pesos didáticos para seleção do modelo campeão. "
       "Em um projeto real, esses pesos devem ser negociados com negócio, "
       "risco, compliance e engenharia."
   ),
}

selection_policy


In [ ]:
interpretability_score = {
   "logistic_regression": 1.00,
   "decision_tree": 0.80,
   "random_forest": 0.50,
}

operational_simplicity_score = {
   "logistic_regression": 1.00,
   "decision_tree": 0.90,
   "random_forest": 0.65,
}

decision_df["interpretability_score"] = (
   decision_df["model_name"]
   .map(interpretability_score)
   .fillna(0.50)
)

decision_df["operational_simplicity_score"] = (
   decision_df["model_name"]
   .map(operational_simplicity_score)
   .fillna(0.50)
)

decision_df["stability_score"] = (
   1 - decision_df["roc_auc_cv_desvio"].clip(lower=0, upper=1)
)

decision_df


In [ ]:
decision_df["technical_score"] = (
   0.35 * decision_df["roc_auc_teste"]
   + 0.25 * decision_df["recall_inadimplente"]
   + 0.15 * decision_df["precision_inadimplente"]
   + 0.15 * decision_df["f1_inadimplente"]
   + 0.10 * decision_df["stability_score"]
)

decision_df["final_score"] = (
   0.75 * decision_df["technical_score"]
   + 0.15 * decision_df["interpretability_score"]
   + 0.10 * decision_df["operational_simplicity_score"]
)

decision_ranking = (
   decision_df
   .sort_values(["final_score", "roc_auc_teste"], ascending=False)
   .reset_index(drop=True)
)

decision_ranking


In [ ]:
champion = decision_ranking.iloc[0].to_dict()

champion_run_id = champion["run_id"]
champion_model_name = champion["model_name"]
champion_run_name = champion["run_name"]

champion_run_id, champion_model_name, champion_run_name


In [ ]:
champion_summary = f"""
# Seleção do modelo campeão

## Contexto

Projeto: risco de crédito 
Tarefa: classificação binária 
Target: inadimplente 
Dataset version: synthetic_credit_v1 
Feature set: core_features_v1 

## Modelo selecionado

- Modelo: {champion_model_name}
- Run name: {champion_run_name}
- Run ID: {champion_run_id}

## Principais métricas

- ROC-AUC teste: {champion["roc_auc_teste"]:.4f}
- ROC-AUC CV média: {champion["roc_auc_cv_media"]:.4f}
- ROC-AUC CV desvio: {champion["roc_auc_cv_desvio"]:.4f}
- Precision inadimplente: {champion["precision_inadimplente"]:.4f}
- Recall inadimplente: {champion["recall_inadimplente"]:.4f}
- F1 inadimplente: {champion["f1_inadimplente"]:.4f}

## Critérios de decisão

A seleção considerou desempenho técnico, estabilidade,
interpretabilidade e simplicidade operacional.

Pesos técnicos:
- ROC-AUC teste: 35%
- Recall inadimplente: 25%
- Precision inadimplente: 15%
- F1 inadimplente: 15%
- Estabilidade: 10%

Score final:
- Score técnico: 75%
- Interpretabilidade: 15%
- Simplicidade operacional: 10%

## Justificativa

O modelo selecionado apresentou o melhor equilíbrio entre desempenho
e critérios operacionais dentro da política de seleção definida para esta aula.

Esta decisão é didática. Em um projeto real, os thresholds e pesos deveriam
ser validados com áreas de negócio, risco, compliance e engenharia.

## Próximo passo

Preparar este modelo para registro no MLflow Model Registry na próxima aula.
"""

print(champion_summary)


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.set_tag(champion_run_id, "selection_status", "champion_candidate")
client.set_tag(champion_run_id, "selected_for_registry", "true")
client.set_tag(champion_run_id, "selection_rule", "weighted_score_v1")
client.set_tag(champion_run_id, "registry_model_name", "risco_credito_classifier")
client.set_tag(champion_run_id, "course_lesson_selection", "aula_05")

mlflow.log_text(
   champion_summary,
   "selection/champion_summary.md",
   run_id=champion_run_id,
)

mlflow.log_dict(
   selection_policy,
   "selection/selection_policy.json",
   run_id=champion_run_id,
)

mlflow.log_table(
   decision_ranking,
   "selection/decision_matrix.json",
   run_id=champion_run_id,
)


In [ ]:
MODEL_ARTIFACT_PATH = "model"

champion_model_uri = f"runs:/{champion_run_id}/{MODEL_ARTIFACT_PATH}"

registry_preparation = {
   "registered_model_name": "risco_credito_classifier",
   "source_run_id": champion_run_id,
   "source_run_name": champion_run_name,
   "model_name": champion_model_name,
   "model_uri": champion_model_uri,
   "intended_stage_next_lesson": "Staging",
   "selection_status": "champion_candidate",
   "notes": (
       "Metadados preparados na Aula 5. "
       "O registro efetivo no Model Registry será feito na Aula 6."
   ),
}

mlflow.log_dict(
   registry_preparation,
   "selection/registry_preparation.json",
   run_id=champion_run_id,
)

registry_preparation


## 6.x - Preparando o backend local com SQLite

mlflow server \
 --backend-store-uri sqlite:///mlflow_registry.db \
 --default-artifact-root ./mlartifacts \
 --host 127.0.0.1 \
 --port 5000


Função auxiliar para montar o modelo campeão


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


def build_estimator(model_name):
   candidates = {
       "logistic_regression": LogisticRegression(
           max_iter=2000,
           class_weight="balanced",
           random_state=RANDOM_STATE,
       ),
       "decision_tree": DecisionTreeClassifier(
           max_depth=5,
           min_samples_leaf=50,
           class_weight="balanced",
           random_state=RANDOM_STATE,
       ),
       "random_forest": RandomForestClassifier(
           n_estimators=100,
           max_depth=8,
           min_samples_leaf=20,
           class_weight="balanced",
           n_jobs=-1,
           random_state=RANDOM_STATE,
       ),
   }

   if model_name not in candidates:
       raise ValueError(f"Modelo não reconhecido: {model_name}")

   return clone(candidates[model_name])


Definindo o nome do modelo registrado

In [ ]:
REGISTERED_MODEL_NAME = "risco_credito_classifier_pyfunc"

CHAMPION_MODEL_NAME = champion_model_name
CHAMPION_MODEL_NAME


Logando e registrando o modelo campeão

In [ ]:
import pandas as pd
import mlflow.sklearn

from mlflow.models import infer_signature
from sklearn.metrics import (
   accuracy_score,
   f1_score,
   precision_score,
   recall_score,
   roc_auc_score,
)
from sklearn.pipeline import Pipeline


mlflow.set_experiment("risco_credito/06_model_registry")

champion_estimator = build_estimator(CHAMPION_MODEL_NAME)

champion_pipe = Pipeline(
   steps=[
       ("prep", make_preprocess()),
       ("model", champion_estimator),
   ]
)

input_example = X_train.head(5).copy()

with mlflow.start_run(run_name="aula06_register_champion") as run:
   champion_pipe.fit(X_train, y_train)

   y_pred = champion_pipe.predict(X_test)
   y_proba = champion_pipe.predict_proba(X_test)[:, 1]

   metrics = {
       "roc_auc_teste": float(roc_auc_score(y_test, y_proba)),
       "accuracy_teste": float(accuracy_score(y_test, y_pred)),
       "precision_inadimplente": float(
           precision_score(y_test, y_pred, zero_division=0)
       ),
       "recall_inadimplente": float(
           recall_score(y_test, y_pred, zero_division=0)
       ),
       "f1_inadimplente": float(
           f1_score(y_test, y_pred, zero_division=0)
       ),
   }

   mlflow.log_params(
       {
           "model_name": CHAMPION_MODEL_NAME,
           "registered_model_name": REGISTERED_MODEL_NAME,
           "dataset_version": "synthetic_credit_v1",
           "feature_set": "core_features_v1",
           "target": TARGET,
           "registry_lesson": "aula_06",
       }
   )

   mlflow.log_metrics(metrics)

   mlflow.set_tags(
       {
           "project": "risco_credito",
           "course_lesson": "aula_06",
           "run_purpose": "register_champion_model",
           "selection_status": "champion_candidate",
           "tracking_backend": "sqlite_local",
       }
   )

   signature = infer_signature(
       input_example,
       champion_pipe.predict(input_example),
   )

   model_info = mlflow.sklearn.log_model(
       sk_model=champion_pipe,
       name="model",
       signature=signature,
       input_example=input_example,
       registered_model_name=REGISTERED_MODEL_NAME,
   )

   run_id = run.info.run_id

print("Run ID:", run_id)
print("Model URI:", model_info.model_uri)


Localizando a versão criada

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

model_versions = client.search_model_versions(
   f"name = '{REGISTERED_MODEL_NAME}'"
)

registered_version = max(
   [mv for mv in model_versions if mv.run_id == run_id],
   key=lambda mv: int(mv.version),
)

print("Registered model:", registered_version.name)
print("Version:", registered_version.version)
print("Run ID:", registered_version.run_id)
print("Source:", registered_version.source)
print("Status:", registered_version.status)


Adicionando descrição e tags ao Registry

In [ ]:
model_description = """
Modelo de classificação de risco de crédito usado no projeto didático do curso.

Este registered model agrupa versões candidatas do classificador de inadimplência.
As versões devem ser promovidas por aliases e tags de ciclo de vida.
"""

version_description = f"""
Versão registrada a partir da run {run_id}.

Modelo selecionado na etapa de model selection do projeto.
Algoritmo: {CHAMPION_MODEL_NAME}
Dataset: synthetic_credit_v1
Feature set: core_features_v1
"""

client.update_registered_model(
   name=REGISTERED_MODEL_NAME,
   description=model_description,
)

client.update_model_version(
   name=REGISTERED_MODEL_NAME,
   version=registered_version.version,
   description=version_description,
)

client.set_registered_model_tag(
   REGISTERED_MODEL_NAME,
   "project",
   "risco_credito",
)

client.set_registered_model_tag(
   REGISTERED_MODEL_NAME,
   "task",
   "binary_classification",
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   registered_version.version,
   "dataset_version",
   "synthetic_credit_v1",
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   registered_version.version,
   "model_name",
   CHAMPION_MODEL_NAME,
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   registered_version.version,
   "lifecycle_status",
   "candidate",
)


Adicionando aliases iniciais

In [ ]:
client.set_registered_model_alias(
   name=REGISTERED_MODEL_NAME,
   alias="candidate",
   version=registered_version.version,
)

client.set_registered_model_alias(
   name=REGISTERED_MODEL_NAME,
   alias="champion_candidate",
   version=registered_version.version,
)

alias_version = client.get_model_version_by_alias(
   REGISTERED_MODEL_NAME,
   "candidate",
)

print("Alias candidate aponta para a versão:", alias_version.version)


Promovendo para staging

In [ ]:
MODEL_VERSION = registered_version.version

client.set_registered_model_alias(
   name=REGISTERED_MODEL_NAME,
   alias="staging",
   version=MODEL_VERSION,
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   MODEL_VERSION,
   "lifecycle_status",
   "staging",
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   MODEL_VERSION,
   "validation_status",
   "pending_business_validation",
)

staging_version = client.get_model_version_by_alias(
   REGISTERED_MODEL_NAME,
   "staging",
)

print("Alias staging aponta para a versão:", staging_version.version)


Validando o modelo em staging

In [ ]:
novo_cliente = pd.DataFrame(
   [
       {
           "idade": 34,
           "renda_mensal": 4200.0,
           "tempo_emprego_anos": 5.0,
           "score_interno": 620,
           "valor_solicitado": 12000.0,
           "num_parcelas": 24,
           "tipo_renda": "CLT",
           "possui_restricao": "nao",
       }
   ]
)

staging_model_uri = f"models:/{REGISTERED_MODEL_NAME}@staging"

staging_model = mlflow.pyfunc.load_model(staging_model_uri)

staging_prediction = staging_model.predict(novo_cliente)

print("Modelo carregado de:", staging_model_uri)
print("Predição:", staging_prediction)


In [ ]:
sklearn_model = mlflow.sklearn.load_model(staging_model_uri)

probabilidade = sklearn_model.predict_proba(novo_cliente)[0, 1]

print(f"Probabilidade estimada de inadimplência: {probabilidade:.1%}")


Promovendo para production

In [ ]:
client.set_registered_model_alias(
   name=REGISTERED_MODEL_NAME,
   alias="production",
   version=MODEL_VERSION,
)

client.set_registered_model_alias(
   name=REGISTERED_MODEL_NAME,
   alias="champion",
   version=MODEL_VERSION,
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   MODEL_VERSION,
   "lifecycle_status",
   "production",
)

client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   MODEL_VERSION,
   "validation_status",
   "approved_for_production",
)

production_version = client.get_model_version_by_alias(
   REGISTERED_MODEL_NAME,
   "production",
)

print("Alias production aponta para a versão:", production_version.version)


Validando a versão de produção antes de servir


In [ ]:
production_model_uri = f"models:/{REGISTERED_MODEL_NAME}@production"

prod_version = client.get_model_version_by_alias(
   REGISTERED_MODEL_NAME,
   "production",
)

print("Modelo:", prod_version.name)
print("Versão de produção:", prod_version.version)
print("Run de origem:", prod_version.run_id)
print("Source:", prod_version.source)

production_model = mlflow.pyfunc.load_model(production_model_uri)

production_prediction = production_model.predict(novo_cliente)

print("Modelo carregado de:", production_model_uri)
print("Predição:", production_prediction)


Arquivando uma versão

In [ ]:
client.set_model_version_tag(
   REGISTERED_MODEL_NAME,
   MODEL_VERSION,
   "archive_policy",
   "keep_until_replaced_by_new_champion",
)


##  7.x - Formato MLmodel


Recuperando o modelo de produção

In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.pyfunc

from pathlib import Path

mlflow.set_tracking_uri("http://127.0.0.1:5000")

REGISTERED_MODEL_NAME = "risco_credito_classifier"

production_model_uri = f"models:/{REGISTERED_MODEL_NAME}@production"

production_model_uri


Baixando o diretório do modelo

In [ ]:
local_model_path = mlflow.artifacts.download_artifacts(
   artifact_uri=production_model_uri
)

local_model_path = Path(local_model_path)

print(local_model_path)


Listando os arquivos do modelo

In [ ]:
for path in sorted(local_model_path.rglob("*")):
   print(path.relative_to(local_model_path))


Abrindo o arquivo MLmodel


In [ ]:
mlmodel_path = local_model_path / "MLmodel"

print(mlmodel_path.read_text(encoding="utf-8"))


Lendo o MLmodel como dicionário

In [ ]:
import yaml

with open(mlmodel_path, encoding="utf-8") as f:
   mlmodel_config = yaml.safe_load(f)

mlmodel_config.keys()



In [ ]:
mlmodel_config.get("flavors")

Carregando com flavor sklearn

In [ ]:
sklearn_model = mlflow.sklearn.load_model(production_model_uri)

novo_cliente = pd.DataFrame(
   [
       {
           "idade": 34,
           "renda_mensal": 4200.0,
           "tempo_emprego_anos": 5.0,
           "score_interno": 620,
           "valor_solicitado": 12000.0,
           "num_parcelas": 24,
           "tipo_renda": "CLT",
           "possui_restricao": "nao",
       }
   ]
)

classe = sklearn_model.predict(novo_cliente)[0]
probabilidade = sklearn_model.predict_proba(novo_cliente)[0, 1]

print("Classe prevista:", classe)
print(f"Probabilidade de inadimplência: {probabilidade:.1%}")


Carregando com flavor pyfunc

In [ ]:
pyfunc_model = mlflow.pyfunc.load_model(production_model_uri)

predicao = pyfunc_model.predict(novo_cliente)

predicao


Inspecionando a signature

In [ ]:
from mlflow.models import get_model_info

model_info = get_model_info(production_model_uri)

print(model_info.signature)


Testando entrada válida

In [ ]:
entrada_valida = X_test.head(3).copy()

pyfunc_model.predict(entrada_valida)


Testando erro por coluna ausente

In [ ]:
entrada_sem_coluna = entrada_valida.drop(columns=["score_interno"])

try:
   pyfunc_model.predict(entrada_sem_coluna)
except Exception as e:
   print(type(e).__name__)
   print(str(e)[:700])


Testando coluna extra

In [ ]:
entrada_com_coluna_extra = entrada_valida.copy()
entrada_com_coluna_extra["canal_origem"] = "app"

pyfunc_model.predict(entrada_com_coluna_extra)


Reforçando o papel do input example

In [ ]:
input_example_path = local_model_path / "input_example.json"
serving_example_path = local_model_path / "serving_input_example.json"

print("input_example existe?", input_example_path.exists())
print("serving_input_example existe?", serving_example_path.exists())
#Tela: Se existir:
if input_example_path.exists():
   print(input_example_path.read_text(encoding="utf-8")[:1000])

if serving_example_path.exists():
   print(serving_example_path.read_text(encoding="utf-8")[:1000])


## Continuação  7

Baixando o modelo sklearn base

In [ ]:
sklearn_model_local_path = mlflow.artifacts.download_artifacts(
   artifact_uri=production_model_uri
)

sklearn_model_local_path


Criando a classe customizada

In [ ]:
import numpy as np
import pandas as pd
import mlflow.pyfunc
import mlflow.sklearn


class CreditRiskPyfuncModel(mlflow.pyfunc.PythonModel):
   def __init__(self, required_columns, default_threshold=0.5):
       self.required_columns = list(required_columns)
       self.default_threshold = float(default_threshold)

   def load_context(self, context):

       self.sklearn_model = mlflow.sklearn.load_model(
           context.artifacts["sklearn_model"]
       )

   def predict(self, context, model_input, params=None):
       if not isinstance(model_input, pd.DataFrame):
           model_input = pd.DataFrame(model_input)

       missing_columns = [
           col for col in self.required_columns
           if col not in model_input.columns
       ]

       if missing_columns:
           raise ValueError(
               f"Colunas obrigatórias ausentes: {missing_columns}"
           )

       threshold = self.default_threshold

       if params is not None and "threshold" in params:
           threshold = float(params["threshold"])

       X = model_input[self.required_columns].copy()

       probabilities = self.sklearn_model.predict_proba(X)[:, 1]
       predicted_classes = (probabilities >= threshold).astype(int)

       return pd.DataFrame(
           {
               "probabilidade_inadimplencia": probabilities,
               "classe_prevista": predicted_classes,
               "limiar_utilizado": np.repeat(threshold, len(X)),
           }
       )


Testando a classe antes de logar

In [ ]:
wrapper_teste = CreditRiskPyfuncModel(
   required_columns=FEATURES,
   default_threshold=0.5,
)

wrapper_teste.sklearn_model = mlflow.sklearn.load_model(
   production_model_uri
)

input_example = X_train.head(5).copy()

output_example = wrapper_teste.predict(
   context=None,
   model_input=input_example,
)

output_example


Criando uma signature para o pyfunc

In [ ]:
from mlflow.models import infer_signature

signature = infer_signature(
   input_example,
   output_example,
   params={"threshold": 0.5},
)

print(signature)


Logando o modelo pyfunc

In [ ]:
PYFUNC_MODEL_NAME = "risco_credito_classifier_pyfunc"

mlflow.set_experiment("risco_credito/07_mlflow_models")

with mlflow.start_run(run_name="aula07_pyfunc_credit_risk_contract") as run:
   pyfunc_model_info = mlflow.pyfunc.log_model(
       name="credit_risk_pyfunc",
       python_model=CreditRiskPyfuncModel(
           required_columns=FEATURES,
           default_threshold=0.5,
       ),
       artifacts={
           "sklearn_model": sklearn_model_local_path,
       },
       signature=signature,
       input_example=input_example,
       registered_model_name=PYFUNC_MODEL_NAME,
       pip_requirements=[
           "mlflow",
           "pandas",
           "numpy",
           "scikit-learn",
           "cloudpickle",
       ],
   )

   mlflow.log_params(
       {
           "source_model_uri": production_model_uri,
           "default_threshold": 0.5,
           "output_format": "probability_label_threshold",
           "registered_model_name": PYFUNC_MODEL_NAME,
       }
   )

   mlflow.set_tags(
       {
           "project": "risco_credito",
           "course_lesson": "aula_07",
           "model_interface": "custom_pyfunc",
           "run_purpose": "custom_inference_contract",
       }
   )

   pyfunc_run_id = run.info.run_id

print("Run ID:", pyfunc_run_id)
print("Model URI:", pyfunc_model_info.model_uri)


Carregando e testando o pyfunc

In [ ]:
loaded_pyfunc = mlflow.pyfunc.load_model(
   pyfunc_model_info.model_uri
)

loaded_pyfunc.predict(novo_cliente)


Testando limiar customizado

In [ ]:
loaded_pyfunc.predict(
   novo_cliente,
   params={"threshold": 0.30},
)


Testando erro controlado

In [ ]:
cliente_invalido = novo_cliente.drop(columns=["score_interno"])

try:
   loaded_pyfunc.predict(cliente_invalido)
except Exception as e:
   print(type(e).__name__)
   print(e)


 Definindo alias para o pyfunc

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

pyfunc_versions = client.search_model_versions(
   f"name = '{PYFUNC_MODEL_NAME}'"
)

pyfunc_registered_version = max(
   [mv for mv in pyfunc_versions if mv.run_id == pyfunc_run_id],
   key=lambda mv: int(mv.version),
)

client.set_registered_model_alias(
   name=PYFUNC_MODEL_NAME,
   alias="candidate",
   version=pyfunc_registered_version.version,
)

client.set_registered_model_alias(
   name=PYFUNC_MODEL_NAME,
   alias="serving_candidate",
   version=pyfunc_registered_version.version,
)

client.set_model_version_tag(
   PYFUNC_MODEL_NAME,
   pyfunc_registered_version.version,
   "interface",
   "probability_label_threshold",
)

print("Versão pyfunc:", pyfunc_registered_version.version)


Carregando o pyfunc pelo Registry


In [ ]:
serving_candidate_uri = f"models:/{PYFUNC_MODEL_NAME}@serving_candidate"

serving_candidate_model = mlflow.pyfunc.load_model(
   serving_candidate_uri
)

serving_candidate_model.predict(novo_cliente)


##  8.x - Garantindo que o MLflow Tracking Server está rodando

mlflow server \
 --backend-store-uri sqlite:///mlflow_registry.db \
 --default-artifact-root ./mlartifacts \
 --host 127.0.0.1 \
 --port 5000


models:/risco_credito_classifier_pyfunc@serving_candidate

Subindo o serving local

macOS/Linux.
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
export MODEL_URI="models:/risco_credito_classifier_pyfunc@serving_candidate"

mlflow models serve \
 -m "$MODEL_URI" \
 -p 5001 \
 --env-manager local

Windows PowerShell.
$env:MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
$env:MODEL_URI = "models:/risco_credito_classifier_pyfunc@serving_candidate"

mlflow models serve -m $env:MODEL_URI -p 5001 --env-manager local


Testando saúde da API com curl

curl http://127.0.0.1:5001/ping 

curl http://127.0.0.1:5001/health


Chamando a API com curl

curl -X POST http://127.0.0.1:5001/invocations \
 -H "Content-Type: application/json" \
 -d '{
   "dataframe_split": {
     "columns": [
       "idade",
       "renda_mensal",
       "tempo_emprego_anos",
       "score_interno",
       "valor_solicitado",
       "num_parcelas",
       "tipo_renda",
       "possui_restricao"
     ],
     "data": [
       [34, 4200.0, 5.0, 620, 12000.0, 24, "CLT", "nao"]
     ]
   },
   "params": {
     "threshold": 0.3
   }
 }'


Construindo a imagem Docker

export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
export MODEL_URI="models:/risco_credito_classifier_pyfunc@serving_candidate"

mlflow models build-docker \
 --model-uri "$MODEL_URI" \
 --name "risco-credito-serving:latest"


Subindo o container

docker run --rm \
 -p 5002:8080 \
 --name risco-credito-api \
 "risco-credito-serving:latest"


Repetindo a inferência no container

curl -X POST http://127.0.0.1:5002/invocations \
 -H "Content-Type: application/json" \
 -d '{
   "dataframe_split": {
     "columns": [
       "idade",
       "renda_mensal",
       "tempo_emprego_anos",
       "score_interno",
       "valor_solicitado",
       "num_parcelas",
       "tipo_renda",
       "possui_restricao"
     ],
     "data": [
       [34, 4200.0, 5.0, 620, 12000.0, 24, "CLT", "nao"]
     ]
   },
   "params": {
     "threshold": 0.3
   }
 }'


In [ ]:
from pathlib import Path
import shutil
import mlflow.sklearn


REGISTERED_MODEL_NAME='risco_credito_classifier_pyfunc'

production_model_uri = f"models:/{REGISTERED_MODEL_NAME}@serving_candidate"

# Carrega o modelo sklearn original
sklearn_model = mlflow.sklearn.load_model(production_model_uri)

# Cria uma pasta limpa para embutir no PyFunc
clean_sklearn_model_path = Path("sklearn_model_for_pyfunc").resolve()

if clean_sklearn_model_path.exists():
    shutil.rmtree(clean_sklearn_model_path)

# Salva novamente como um modelo sklearn MLflow autocontido
mlflow.sklearn.save_model(
    sk_model=sklearn_model,
    path=str(clean_sklearn_model_path),
)

print("clean_sklearn_model_path:", clean_sklearn_model_path)
print("Existe MLmodel?", (clean_sklearn_model_path / "MLmodel").exists())

Criando um smoke test da API

In [ ]:
import json
import sys

import requests


API_URL = "http://127.0.0.1:5001"


def check_health():
   response = requests.get(f"{API_URL}/ping", timeout=10)
   response.raise_for_status()


def check_prediction_contract():
   payload = {
       "dataframe_split": {
           "columns": [
               "idade",
               "renda_mensal",
               "tempo_emprego_anos",
               "score_interno",
               "valor_solicitado",
               "num_parcelas",
               "tipo_renda",
               "possui_restricao",
           ],
           "data": [
               [34, 4200.0, 5.0, 620, 12000.0, 24, "CLT", "nao"],
           ],
       },
       "params": {
           "threshold": 0.3,
       },
   }

   response = requests.post(
       f"{API_URL}/invocations",
       headers={"Content-Type": "application/json"},
       data=json.dumps(payload),
       timeout=30,
   )
   response.raise_for_status()

   body = response.json()

   if "predictions" not in body:
       raise AssertionError("Resposta não contém a chave 'predictions'.")

   predictions = body["predictions"]

   if not isinstance(predictions, list) or len(predictions) != 1:
       raise AssertionError("Resposta deveria conter uma predição.")

   prediction = predictions[0]

   required_fields = {
       "probabilidade_inadimplencia",
       "classe_prevista",
       "limiar_utilizado",
   }

   missing_fields = required_fields - set(prediction.keys())

   if missing_fields:
       raise AssertionError(
           f"Campos ausentes na resposta: {sorted(missing_fields)}"
       )

   prob = prediction["probabilidade_inadimplencia"]
   classe = prediction["classe_prevista"]
   threshold = prediction["limiar_utilizado"]

   if not 0 <= prob <= 1:
       raise AssertionError("Probabilidade fora do intervalo [0, 1].")

   if classe not in [0, 1]:
       raise AssertionError("Classe prevista deve ser 0 ou 1.")

   if abs(threshold - 0.3) > 1e-9:
       raise AssertionError("Limiar retornado não corresponde ao enviado.")

   return body


if __name__ == "__main__":
   try:
       check_health()
       body = check_prediction_contract()
       print("Smoke test passou.")
       print(json.dumps(body, indent=2, ensure_ascii=False))
   except Exception as exc:
       print("Smoke test falhou.")
       print(type(exc).__name__)
       print(exc)
       sys.exit(1)


Testando erro esperado

In [ ]:
def check_invalid_payload():
   payload = {
       "dataframe_split": {
           "columns": [
               "idade",
               "renda_mensal",
               "tempo_emprego_anos",
               # score_interno removido de propósito
               "valor_solicitado",
               "num_parcelas",
               "tipo_renda",
               "possui_restricao",
           ],
           "data": [
               [34, 4200.0, 5.0, 12000.0, 24, "CLT", "nao"],
           ],
       },
       "params": {
           "threshold": 0.3,
       },
   }

   response = requests.post(
       f"{API_URL}/invocations",
       headers={"Content-Type": "application/json"},
       data=json.dumps(payload),
       timeout=30,
   )

   if response.status_code < 400:
       raise AssertionError(
           "Payload inválido deveria retornar erro, mas não retornou."
       )

   print("Erro esperado para payload inválido:")
   print(response.status_code)
   print(response.text[:500])


In [ ]:
check_invalid_payload()